# Permitted Kashmiri YouTube playlist ingestion
Downloads the creator-authorized playlist for dataset preparation. This job does not train on unreviewed transcripts.

In [ ]:
!pip install -q yt-dlp webvtt-py
!ffmpeg -version | head -1

In [ ]:
from pathlib import Path
import datetime, json, subprocess

PLAYLIST = 'https://youtube.com/playlist?list=PLSbPmiCa4iEEkZBX7lkLZdRdefcjLR1QH'
root = Path('/kaggle/working/permitted-kashmiri-playlist')
media = root / 'media'
media.mkdir(parents=True, exist_ok=True)
provenance = {
  'source_url': PLAYLIST,
  'playlist_id': 'PLSbPmiCa4iEEkZBX7lkLZdRdefcjLR1QH',
  'permission_basis': 'Uploader/rights-holder permission reported by project owner',
  'permission_record_status': 'External written evidence must be retained by project owner',
  'recorded_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'allowed_use': 'Kashmiri speech dataset preparation and model training',
  'review_required': True
}
(root / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
cmd = [
  'yt-dlp', '--ignore-errors', '--yes-playlist', '--no-overwrites',
  '--write-info-json', '--write-description', '--write-subs', '--write-auto-subs',
  '--sub-langs', 'ks.*,en.*', '--sub-format', 'vtt',
  '--extract-audio', '--audio-format', 'flac', '--audio-quality', '0',
  '--output', str(media / '%(playlist_index)03d-%(id)s.%(ext)s'), PLAYLIST
]
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(root / 'download.log').write_text(result.stdout, encoding='utf-8')
print(result.stdout[-12000:])
print('yt-dlp exit:', result.returncode)

In [ ]:
import csv, json, subprocess
rows = []
for info_path in sorted(media.glob('*.info.json')):
    info = json.loads(info_path.read_text(encoding='utf-8'))
    stem = info_path.name.removesuffix('.info.json')
    audio = next(iter(media.glob(stem + '.*')), None)
    captions = sorted(str(p.name) for p in media.glob(stem + '*.vtt'))
    rows.append({
      'video_id': info.get('id'), 'title': info.get('title'),
      'uploader': info.get('uploader'), 'channel_id': info.get('channel_id'),
      'duration': info.get('duration'), 'webpage_url': info.get('webpage_url'),
      'audio_file': audio.name if audio and audio.suffix == '.flac' else '',
      'caption_files': '|'.join(captions),
      'review_status': 'pending', 'training_eligible': 'no'
    })
fields = list(rows[0]) if rows else ['video_id','title','uploader','channel_id','duration','webpage_url','audio_file','caption_files','review_status','training_eligible']
with (root / 'inventory.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
print('inventoried videos:', len(rows))
print('FLAC files:', len(list(media.glob('*.flac'))))
print('caption files:', len(list(media.glob('*.vtt'))))
assert rows, 'No playlist metadata was downloaded'